# Loading libraries


In [ ]:
import pandas as pd
import numpy as np

# Loading datasets

In [ ]:
predictions = pd.read_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/results/metrics/top_putative_positives.csv")

In [ ]:
af = pd.read_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/Populational_Data/Populational_variants_AF.txt", sep="\t",
                 names=['User_input', 'AF_EUR', 'AF_EAS', 'AF_AMR', 'AFR_SAS', 'AF_AFR'])


variants_decoy = pd.read_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/Populational_Data/Populational_variants_withoutAF.txt", sep="\t")
variants_decoy[['User_input', 'Gene']] = variants_decoy['User_input'].str.rsplit(':', n=1, expand=True)

In [ ]:
merged_df = pd.merge(variants_decoy, af, on='User_input', how='left')


In [ ]:
merged_df['User_input'] = merged_df['User_input'] + ':' + merged_df['Gene']
merged_df = merged_df.drop(columns=['Gene', 'AF_AMR', 'AFR_SAS'])

In [ ]:
cols = ["AF_EUR", "AF_EAS", "AF_AFR"]

merged_df[cols] = (
    merged_df[cols]
    .replace(".", 0)
    .apply(pd.to_numeric)
)

In [ ]:
def calculate_pairwise_fst(p_i, p_j):
    p_i = np.asarray(p_i)
    p_j = np.asarray(p_j)
    h_s = (2 * p_i * (1 - p_i) + 2 * p_j * (1 - p_j)) / 2.0
    p_bar = (p_i + p_j) / 2.0
    h_t = 2 * p_bar * (1 - p_bar)
    fst = np.zeros_like(h_t, dtype=float)
    valid = h_t > 0
    np.divide((h_t - h_s), h_t, out=fst, where=valid)
    return np.clip(fst, 0, 0.999999)

def calculate_global_fst(freq_arrays):
    k = len(freq_arrays)
    h_sub = sum(2 * p * (1 - p) for p in freq_arrays) / k
    p_bar = sum(freq_arrays) / k
    h_total = 2 * p_bar * (1 - p_bar)
    fst = np.zeros_like(h_total, dtype=float)
    valid = h_total > 0
    np.divide((h_total - h_sub), h_total, out=fst, where=valid)
    return np.clip(fst, 0, 0.999999)

def calculate_pbs(p_target, p_sister, p_outgroup):
    fst_12 = calculate_pairwise_fst(p_target, p_sister)
    fst_13 = calculate_pairwise_fst(p_target, p_outgroup)
    fst_23 = calculate_pairwise_fst(p_sister, p_outgroup)
    t_12 = -np.log(1 - fst_12)
    t_13 = -np.log(1 - fst_13)
    t_23 = -np.log(1 - fst_23)
    return (t_12 + t_13 - t_23) / 2.0

In [ ]:
merged_df['Global_FST'] = calculate_global_fst([merged_df['AF_EUR'], merged_df['AF_EAS'], merged_df['AF_AFR']])

merged_df['PBS_EUR'] = calculate_pbs(merged_df['AF_EUR'], merged_df['AF_EAS'], merged_df['AF_AFR'])
merged_df['PBS_EAS'] = calculate_pbs(merged_df['AF_EAS'], merged_df['AF_EUR'], merged_df['AF_AFR'])
merged_df['PBS_AFR'] = calculate_pbs(merged_df['AF_AFR'], merged_df['AF_EUR'], merged_df['AF_EAS'])

In [ ]:
def determine_significant_population(row, min_global_fst=0.02, min_pbs=0.05):
    if row['Global_FST'] < min_global_fst:
        return 'Not significant'

    if (row['AF_EUR'] > row['AF_EAS']) and (row['AF_EUR'] > row['AF_AFR']) and (row['PBS_EUR'] >= min_pbs):
        return 'EUR'

    if (row['AF_EAS'] > row['AF_EUR']) and (row['AF_EAS'] > row['AF_AFR']) and (row['PBS_EAS'] >= min_pbs):
        return 'EAS'

    if (row['AF_AFR'] > row['AF_EUR']) and (row['AF_AFR'] > row['AF_EAS']) and (row['PBS_AFR'] >= min_pbs):
        return 'AFR'

    return 'Ambiguous divergence'

In [ ]:
merged_df['prevalent_pop'] = merged_df.apply(determine_significant_population, axis=1)

In [ ]:
predictions_af = pd.merge(predictions, merged_df, on='User_input', how='left')

In [ ]:
predictions_af.to_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/results/metrics/top_putative_positives_populations.csv", na_rep="NA", index=False)

In [ ]:
predictions_af.groupby('prevalent_pop').size()

In [ ]:
predictions_af_60 = predictions_af[predictions_af['predicted_probability'] > 0.6]
predictions_af_60.groupby('prevalent_pop').size()

In [ ]:
predictions_af_50 = predictions_af[predictions_af['predicted_probability'] > 0.5]
predictions_af_50.groupby('prevalent_pop').size()